In [0]:
# Fetch Apple stock prices for the last year using ld API
import lseg.data as ld
from datetime import datetime, timedelta

# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
%pip install lseg-data --quiet

import sys
sys.path.append('/Workspace/Users/nvaldez@tec.mx/finanzas_2026_agosto')

# Force reload of modules
import importlib
import analisis_precios_lseg.data.downloader
import analisis_precios_lseg.data.portfolio
importlib.reload(analisis_precios_lseg.data.downloader)
importlib.reload(analisis_precios_lseg.data.portfolio)

# Import after reload
from analisis_precios_lseg.data.portfolio import PortfolioDownloader

# Definir una lista de RICs para el portafolio
rics = ['AAPL.O', 'MSFT.O', 'GOOGL.O', 'AMZN.O', 'TSLA.O']

# Crear el downloader de portafolio
portfolio = PortfolioDownloader(
    rics=rics,
    app_key=APP_KEY,
    username=USERNAME,
    password=PASSWORD,
    use_secrets=False
)

# Descargar datos de los últimos 180 días
data = portfolio.download_portfolio(days=180)

# Calcular retornos diarios
returns = portfolio.get_returns(log_returns=False)


In [0]:
display(data)

In [0]:
import numpy as np
import pandas as pd

# ==============================================================================
# SECCIÓN 1: INSUMOS DEL MODELO BLACK-LITTERMAN
# ==============================================================================

# Lista de activos en el universo de inversión (N = 4)
assets = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
num_assets = len(assets)

# ------------------------------------------------------------------------------
# 1.1 Insumos del Equilibrio de Mercado (Prior)
# ------------------------------------------------------------------------------

# Pesos de Capitalización de Mercado (w_mkt): Vector N x 1 (deben sumar 1.0)
w_mkt = np.array([0.35, 0.30, 0.20, 0.15])

# Matriz de Covarianzas (Sigma): Matriz N x N (anualizada)
# Ejemplo con matriz simétrica positiva definida simulada
cov_data = [
    [0.0625, 0.0350, 0.0300, 0.0320],
    [0.0350, 0.0529, 0.0280, 0.0310],
    [0.0300, 0.0280, 0.0484, 0.0290],
    [0.0320, 0.0310, 0.0290, 0.0576]
]
Sigma = pd.DataFrame(cov_data, index=assets, columns=assets).values

# Coeficiente de Aversión al Riesgo del Mercado (Lambda / Risk Aversion)
# Típicamente entre 2.0 y 4.0 para mercados globales
lmbda = 2.5

# ------------------------------------------------------------------------------
# 1.2 Insumos de las Opiniones del Inversionista (Views)
# ------------------------------------------------------------------------------
# Supongamos K = 2 opiniones:
#   View 1 (Absoluta): AAPL tendrá un rendimiento anualizado del 12%.
#   View 2 (Relativa): MSFT superará a GOOGL por un 3% (0.03).

# Vector de retornos de las Views (Q): Vector K x 1
Q = np.array([0.12, 0.03])

# Matriz de Asignación/Dominio (P): Matriz K x N
#   Fila 1 (AAPL = 12%):   [1,  0,  0, 0]
#   Fila 2 (MSFT > GOOGL): [0,  1, -1, 0]
P = np.array([
    [1.0,  0.0,  0.0, 0.0],
    [0.0,  1.0, -1.0, 0.0]
])

# Matriz de Incertidumbre de las Views (Omega): Matriz Diagonal K x K
# Opción A (Específica): Definida manualmente según la confianza en cada view.
Omega = np.diag([0.002, 0.001])

# Opción B (Heurística de He & Litterman): Descomentar para estimar mediante P * Sigma * P^T
# tau_temp = 0.05
# Omega = np.diag(np.diag(tau_temp * P @ Sigma @ P.T))

# ------------------------------------------------------------------------------
# 1.3 Parámetro de Calibración
# ------------------------------------------------------------------------------
# Tau (tau): Escalar que pondera la incertidumbre del equilibrio (típicamente 0.01 a 0.05)
tau = 0.05

# ==============================================================================
# SECCIÓN 2: LÓGICA DE CÁLCULO Y FUNCIONES
# ==============================================================================

def calculate_implied_returns(
    lmbda: float, 
    Sigma: np.ndarray, 
    w_mkt: np.ndarray
) -> np.ndarray:
    """Calcula el vector de retornos implícitos de equilibrio (Pi) mediante Optimización Inversa."""
    return lmbda * (Sigma @ w_mkt)


def black_litterman_posterior(
    Pi: np.ndarray,
    Sigma: np.ndarray,
    P: np.ndarray,
    Q: np.ndarray,
    Omega: np.ndarray,
    tau: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula el vector de retornos a posteriori (mu_BL) y la matriz de covarianza 
    ajustada (Sigma_BL) usando Bayes.
    """
    tau_Sigma_inv = np.linalg.inv(tau * Sigma)
    Omega_inv = np.linalg.inv(Omega)

    # Matriz de precisión posterior
    M = np.linalg.inv(tau_Sigma_inv + P.T @ Omega_inv @ P)

    # Vector de retornos esperados ajustados (mu_BL)
    mu_BL = M @ (tau_Sigma_inv @ Pi + P.T @ Omega_inv @ Q)

    # Matriz de covarianza posterior ajustada por la incertidumbre de la estimación
    Sigma_BL = Sigma + M

    return mu_BL, Sigma_BL


def optimize_portfolio_weights(
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    lmbda: float
) -> np.ndarray:
    """Calcula los pesos óptimos no restringidos basados en el marco de Markowitz."""
    return np.linalg.inv(lmbda * Sigma) @ mu

# ==============================================================================
# SECCIÓN 3: EJECUCIÓN Y RESULTADOS
# ==============================================================================

# 1. Retornos Implícitos de Mercado (Prior)
Pi = calculate_implied_returns(lmbda, Sigma, w_mkt)

# 2. Actualización Bayesiana de Black-Litterman
mu_BL, Sigma_BL = black_litterman_posterior(Pi, Sigma, P, Q, Omega, tau)

# 3. Optimización para obtener pesos finales (w_BL)
w_BL = optimize_portfolio_weights(mu_BL, Sigma_BL, lmbda)


In [0]:
# Visualización comparativa de resultados
df_results = pd.DataFrame({
    'Peso Mercado (w_mkt)': w_mkt,
    'Retorno Implícito (Pi)': Pi,
    'Retorno BL (mu_BL)': mu_BL,
    'Peso Final BL (w_BL)': w_BL,
    'Diferencia (Tilt)': w_BL - w_mkt
}, index=assets)



In [0]:
# Formatear la salida del dataframe a porcentajes para facilitar lectura
df_formatted = df_results.style.format('{:.2%}')
df_formatted

In [0]:
# ==============================================================================
# SECCIÓN 1: INSUMOS DEL MODELO MARKOWITZ
# ==============================================================================

# Universo de activos (N = 4)
assets = ['AAPL.O', 'MSFT.O', 'GOOGL.O', 'AMZN.O', 'TSLA.O']
num_assets = len(assets)

# ------------------------------------------------------------------------------
# 1.1 Insumos de Retornos Esperados (mu)
# ------------------------------------------------------------------------------
# Vector N x 1 con el rendimiento medio proyectado (anualizado) para cada activo.
# Calculado a partir de retornos históricos anualizados (252 días de trading)
mu_data = returns.mean() * 252
mu = mu_data.values

# ------------------------------------------------------------------------------
# 1.2 Insumos de Matriz de Covarianzas (Sigma)
# ------------------------------------------------------------------------------
# Matriz N x N simétrica y positiva definida de covarianzas entre activos (anualizada).
# Calculada a partir de retornos históricos y anualizada (252 días de trading)
cov_data = returns.cov() * 252
Sigma = cov_data.values


In [0]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize


# ------------------------------------------------------------------------------
# 1.3 Parámetro del Entorno (Risk-Free Rate)
# ------------------------------------------------------------------------------
# Tasa libre de riesgo anualizada para el cálculo del Ratio de Sharpe
risk_free_rate = 0.04


# ==============================================================================
# SECCIÓN 2: LÓGICA DE CÁLCULO Y FUNCIONES DE OPTIMIZACIÓN
# ==============================================================================

def portfolio_performance(
    weights: np.ndarray, 
    mu: np.ndarray, 
    Sigma: np.ndarray
) -> tuple[float, float]:
    """Calcula el retorno esperado y la volatilidad (riesgo) de un portafolio."""
    portfolio_return = float(np.sum(weights * mu))
    portfolio_volatility = float(np.sqrt(weights.T @ Sigma @ weights))
    return portfolio_return, portfolio_volatility


def min_volatility_objective(weights: np.ndarray, Sigma: np.ndarray) -> float:
    """Función objetivo: Varianza del portafolio a minimizar."""
    return weights.T @ Sigma @ weights


def negative_sharpe_objective(
    weights: np.ndarray, 
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    rf: float
) -> float:
    """Función objetivo: Negativo del Sharpe Ratio (para maximizarlo)."""
    p_ret, p_vol = portfolio_performance(weights, mu, Sigma)
    return -(p_ret - rf) / p_vol


def optimize_markowitz(
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    rf: float = 0.0,
    target: str = 'max_sharpe',
    allow_short: bool = False
) -> np.ndarray:
    """
    Resuelve el problema de optimización cuadrática de Markowitz.
    
    Parameters:
        mu: Vector de retornos esperados (N x 1)
        Sigma: Matriz de covarianza (N x N)
        rf: Tasa libre de riesgo
        target: 'max_sharpe' para máximo Sharpe Ratio o 'min_vol' para mínima varianza
        allow_short: Si es False, impone restricción de pesos no negativos (w >= 0)
    """
    n = len(mu)
    init_weights = np.ones(n) / n  # Pesos iniciales equi-ponderados
    
    # Restricción: La suma de pesos debe ser igual a 1.0 (Full Investment)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    
    # Limites para cada peso (Longearse únicamente vs Permitir Ventas en Corto)
    bounds = None if allow_short else tuple((0.0, 1.0) for _ in range(n))

    if target == 'max_sharpe':
        objective_fn = lambda w: negative_sharpe_objective(w, mu, Sigma, rf)
    elif target == 'min_vol':
        objective_fn = lambda w: min_volatility_objective(w, Sigma)
    else:
        raise ValueError("El parámetro target debe ser 'max_sharpe' o 'min_vol'.")

    result = minimize(
        fun=objective_fn,
        x0=init_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise RuntimeError(f"La optimización falló: {result.message}")

    return result.x


# ==============================================================================
# SECCIÓN 3: EJECUCIÓN Y RESULTADOS
# ==============================================================================

# 1. Portafolio de Máximo Sharpe Ratio (Largo únicamente: w >= 0)
w_max_sharpe = optimize_markowitz(mu, Sigma, rf=risk_free_rate, target='max_sharpe', allow_short=False)

# 2. Portafolio de Mínima Varianza Global (GMV)
w_min_vol = optimize_markowitz(mu, Sigma, rf=risk_free_rate, target='min_vol', allow_short=False)

# 3. Métricas de Rendimiento
ret_sharpe, vol_sharpe = portfolio_performance(w_max_sharpe, mu, Sigma)
ret_minvol, vol_minvol = portfolio_performance(w_min_vol, mu, Sigma)

# Tabla Comparativa de Resultados
df_results = pd.DataFrame({
    'Retorno Esperado Individual (mu)': mu,
    'Peso Max Sharpe (w*)': w_max_sharpe,
    'Peso Min Varianza (w_gmv)': w_min_vol
}, index=assets)

print("=== PORTAFOLIOS ÓPTIMOS DE MARKOWITZ ===")
print(df_results.style.format('{:.2%}').to_string())

print("\n=== PERFIL DE RIESGO Y RETORNO ===")
print(f"Max Sharpe  -> Retorno: {ret_sharpe:.2%}, Volatilidad: {vol_sharpe:.2%}, Sharpe Ratio: {(ret_sharpe - risk_free_rate)/vol_sharpe:.2f}")
print(f"Min Vol     -> Retorno: {ret_minvol:.2%}, Volatilidad: {vol_minvol:.2%}, Sharpe Ratio: {(ret_minvol - risk_free_rate)/vol_minvol:.2f}")

In [0]:
# Vector de pesos del portafolio óptimo de Markowitz (Máximo Sharpe Ratio)
w_optimo = w_max_sharpe

print("Vector de pesos del portafolio óptimo (Max Sharpe):")
print(w_optimo)

# Mostrar en formato más legible con nombres de activos
df_pesos = pd.DataFrame({
    'Activo': assets,
    'Peso': w_optimo
})
print("\n" + df_pesos.to_string(index=False))

In [0]:
# Obtener información sectorial de cada activo usando LSEG Data API
import lseg.data as ld

# Campos para obtener información sectorial
fields = [
    'TR.TRBCEconomicSector',           # Sector Económico TRBC
    'TR.TRBCEconomicSectorCode',        # Código del Sector
    'TR.TRBCBusinessSector',            # Sector de Negocio
    'TR.TRBCIndustryGroup',             # Grupo Industrial
    'TR.TRBCIndustry',                  # Industria
    'TR.GICSSector',                    # Sector GICS
    'TR.GICSIndustryGroup',             # Grupo Industrial GICS
    'TR.GICSIndustry',                  # Industria GICS
    'TR.CompanyMarketCap'               # Capitalización de Mercado
]

# Obtener datos sectoriales para todos los RICs
df_sectorial = ld.get_data(
    universe=rics,
    fields=fields
)

display(df_sectorial)

In [0]:
# Crear matriz de pesos de activos por sector
import numpy as np
import pandas as pd

# Usar clasificación GICS (más estándar en la industria)
df_sectorial['Sector'] = df_sectorial['GICS Sector Name']
df_sectorial['Market_Cap'] = df_sectorial['Company Market Cap']

# Calcular el peso de cada activo dentro de su sector
# (Market Cap del activo / Total Market Cap del sector)
sector_totals = df_sectorial.groupby('Sector')['Market_Cap'].sum()

# Crear una función para calcular el peso dentro del sector
df_sectorial['Peso_en_Sector'] = df_sectorial.apply(
    lambda row: row['Market_Cap'] / sector_totals[row['Sector']], 
    axis=1
)

# Crear matriz pivot: Activos (filas) x Sectores (columnas)
matriz_sectorial = df_sectorial.pivot_table(
    index='Instrument',
    columns='Sector',
    values='Peso_en_Sector',
    fill_value=0
)

print("=== MATRIZ DE PESOS DE ACTIVOS POR SECTOR ===")
print("\nPeso de cada activo dentro de su sector (basado en Market Cap)")
print("1.0 = 100% del sector, 0.5 = 50% del sector\n")
display(matriz_sectorial.style.format('{:.2%}'))

# Mostrar resumen de capitalización por sector
print("\n=== CAPITALIZACIÓN DE MERCADO POR SECTOR ===")
sector_summary = df_sectorial.groupby('Sector').agg({
    'Market_Cap': 'sum',
    'Instrument': 'count'
}).rename(columns={'Instrument': 'Número_Activos'})

sector_summary['Market_Cap_Trillions'] = sector_summary['Market_Cap'] / 1e12
sector_summary = sector_summary[['Número_Activos', 'Market_Cap_Trillions']]

display(sector_summary.style.format({'Market_Cap_Trillions': '${:.2f}T'}))

# Tabla detallada: Activo, Sector, Market Cap, Peso en Sector
print("\n=== DETALLE POR ACTIVO ===")
df_detalle = df_sectorial[['Instrument', 'Sector', 'Market_Cap', 'Peso_en_Sector']].copy()
df_detalle['Market_Cap_Billions'] = df_detalle['Market_Cap'] / 1e9
df_detalle = df_detalle[['Instrument', 'Sector', 'Market_Cap_Billions', 'Peso_en_Sector']]

display(df_detalle.style.format({
    'Market_Cap_Billions': '${:.2f}B',
    'Peso_en_Sector': '{:.2%}'
}))

Pesos objetivo por sector

In [0]:
# ==============================================================================
# ANÁLISIS DE EVOLUCIÓN DE PESOS DEL PORTAFOLIO EN EL TIEMPO
# ==============================================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# 1. CONFIGURACIÓN INICIAL
# ------------------------------------------------------------------------------

# Capital inicial hace 180 días
capital_inicial = 1_000_000  # $1,000,000 USD

# Pesos óptimos del portafolio de Markowitz (Max Sharpe)
print("=== ASIGNACIÓN INICIAL (Portafolio Óptimo de Markowitz) ===")
df_asignacion_inicial = pd.DataFrame({
    'Activo': assets,
    'Peso_Optimo': w_optimo,
    'Capital_Asignado': w_optimo * capital_inicial
})
print(df_asignacion_inicial.to_string(index=False))

# ------------------------------------------------------------------------------
# 2. CÁLCULO DE ACCIONES COMPRADAS (DÍA 0)
# ------------------------------------------------------------------------------

# Obtener el primer día de precios (hace 180 días)
precios_iniciales = data.iloc[0]

# Calcular número de acciones compradas de cada activo
acciones_compradas = {}
for i, activo in enumerate(assets):
    capital_asignado = w_optimo[i] * capital_inicial
    precio_inicial = precios_iniciales[activo]
    num_acciones = capital_asignado / precio_inicial
    acciones_compradas[activo] = num_acciones

print("\n=== ACCIONES COMPRADAS (DÍA 0) ===")
df_acciones = pd.DataFrame({
    'Activo': assets,
    'Precio_Inicial': [precios_iniciales[a] for a in assets],
    'Capital_Asignado': w_optimo * capital_inicial,
    'Acciones_Compradas': [acciones_compradas[a] for a in assets]
})
print(df_acciones.to_string(index=False))

# ------------------------------------------------------------------------------
# 3. EVOLUCIÓN DIARIA DE PESOS DEL PORTAFOLIO
# ------------------------------------------------------------------------------

# Crear dataframe con la evolución de valores de mercado y pesos
df_evolucion = data.copy()

# Calcular el valor de mercado de cada posición cada día
for activo in assets:
    df_evolucion[f'{activo}_Valor'] = df_evolucion[activo] * acciones_compradas[activo]

# Calcular el valor total del portafolio cada día
columnas_valor = [f'{activo}_Valor' for activo in assets]
df_evolucion['Valor_Total_Portafolio'] = df_evolucion[columnas_valor].sum(axis=1)

# Calcular los pesos dinámicos (weights) de cada activo
for activo in assets:
    df_evolucion[f'{activo}_Peso'] = df_evolucion[f'{activo}_Valor'] / df_evolucion['Valor_Total_Portafolio']

# ------------------------------------------------------------------------------
# 4. AGREGAR INFORMACIÓN SECTORIAL Y CALCULAR PESOS POR SECTOR
# ------------------------------------------------------------------------------

# Crear un mapeo de activo a sector
activo_a_sector = df_sectorial.set_index('Instrument')['Sector'].to_dict()

# Para cada activo, agregar su sector
for activo in assets:
    sector = activo_a_sector.get(activo, 'Unknown')
    df_evolucion[f'{activo}_Sector'] = sector

# Calcular el peso total de cada sector cada día
sectores_unicos = df_sectorial['Sector'].unique()

for sector in sectores_unicos:
    # Identificar qué activos pertenecen a este sector
    activos_en_sector = [a for a in assets if activo_a_sector.get(a) == sector]
    
    # Sumar los pesos de todos los activos en este sector
    columnas_peso_sector = [f'{activo}_Peso' for activo in activos_en_sector]
    df_evolucion[f'Peso_Sector_{sector}'] = df_evolucion[columnas_peso_sector].sum(axis=1)

print("\n=== ESTRUCTURA DEL DATAFRAME DE EVOLUCIÓN ===")
print(f"Dimensiones: {df_evolucion.shape}")
print(f"Periodo: {df_evolucion.index[0]} a {df_evolucion.index[-1]}")
print(f"\nColumnas creadas:")
print(f"  - Precios originales: {assets}")
print(f"  - Valores de mercado: {columnas_valor}")
print(f"  - Pesos dinámicos: {[f'{a}_Peso' for a in assets]}")
print(f"  - Pesos por sector: {[f'Peso_Sector_{s}' for s in sectores_unicos]}")

# ------------------------------------------------------------------------------
# 5. RESUMEN DE ESTADO ACTUAL (DÍA MÁS RECIENTE)
# ------------------------------------------------------------------------------

print("\n=== ESTADO ACTUAL DEL PORTAFOLIO (Último día de datos) ===")
ultimo_dia = df_evolucion.iloc[-1]

df_estado_actual = pd.DataFrame({
    'Activo': assets,
    'Sector': [activo_a_sector[a] for a in assets],
    'Precio_Actual': [ultimo_dia[a] for a in assets],
    'Acciones': [acciones_compradas[a] for a in assets],
    'Valor_Mercado': [ultimo_dia[f'{a}_Valor'] for a in assets],
    'Peso_Actual': [ultimo_dia[f'{a}_Peso'] for a in assets],
    'Peso_Inicial': w_optimo,
    'Drift': [ultimo_dia[f'{a}_Peso'] - w_optimo[i] for i, a in enumerate(assets)]
})

display(df_estado_actual.style.format({
    'Precio_Actual': '${:,.2f}',
    'Acciones': '{:,.2f}',
    'Valor_Mercado': '${:,.2f}',
    'Peso_Actual': '{:.2%}',
    'Peso_Inicial': '{:.2%}',
    'Drift': '{:+.2%}'
}))

print(f"\nValor Total del Portafolio: ${ultimo_dia['Valor_Total_Portafolio']:,.2f}")
print(f"Retorno Total: {(ultimo_dia['Valor_Total_Portafolio'] / capital_inicial - 1) * 100:.2f}%")

# Resumen por Sector
print("\n=== PESOS POR SECTOR (Estado Actual) ===")
df_sectores_actual = pd.DataFrame({
    'Sector': sectores_unicos,
    'Peso_Actual': [ultimo_dia[f'Peso_Sector_{s}'] for s in sectores_unicos]
}).sort_values('Peso_Actual', ascending=False)

display(df_sectores_actual.style.format({'Peso_Actual': '{:.2%}'}))

# Guardar el dataframe de evolución para análisis posteriores
print("\n=== DATAFRAME 'df_evolucion' CREADO ===")
print("Este dataframe contiene la evolución diaria de:")
print("  - Precios de cada activo")
print("  - Valores de mercado de cada posición")
print("  - Pesos dinámicos de cada activo")
print("  - Pesos agregados por sector")
print("  - Valor total del portafolio")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Crear figura con dos subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ------------------------------------------------------------------------------
# SUBPLOT 1: Evolución de Pesos por Activo
# ------------------------------------------------------------------------------
ax1 = axes[0]

# Filtrar datos sin NaT
df_plot = df_evolucion[df_evolucion.index.notna()].copy()

for activo in assets:
    if activo != 'TSLA.O':  # Excluir TSLA porque tiene peso ~0
        ax1.plot(df_plot.index, df_plot[f'{activo}_Peso'], 
                label=activo, linewidth=2, alpha=0.8)

# Líneas horizontales para pesos iniciales
for i, activo in enumerate(assets):
    if activo != 'TSLA.O' and w_optimo[i] > 0.01:
        ax1.axhline(y=w_optimo[i], color='gray', linestyle='--', 
                   alpha=0.3, linewidth=1)

ax1.set_title('Evolución de Pesos del Portafolio por Activo (180 días)', 
             fontsize=14, fontweight='bold')
ax1.set_xlabel('Fecha', fontsize=12)
ax1.set_ylabel('Peso del Activo', fontsize=12)
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

# Formato de fechas en eje x
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ------------------------------------------------------------------------------
# SUBPLOT 2: Evolución de Pesos por Sector
# ------------------------------------------------------------------------------
ax2 = axes[1]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, sector in enumerate(sectores_unicos):
    ax2.fill_between(df_plot.index, 
                     df_plot[f'Peso_Sector_{sector}'],
                     label=sector, alpha=0.6, color=colors[i])

ax2.set_title('Evolución de Pesos del Portafolio por Sector (180 días)', 
             fontsize=14, fontweight='bold')
ax2.set_xlabel('Fecha', fontsize=12)
ax2.set_ylabel('Peso del Sector', fontsize=12)
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
ax2.set_ylim([0, 1])

# Formato de fechas en eje x
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# GRÁFICO 3: Valor Total del Portafolio en el Tiempo
# ------------------------------------------------------------------------------
fig2, ax3 = plt.subplots(figsize=(14, 5))

ax3.plot(df_plot.index, df_plot['Valor_Total_Portafolio'], 
        color='darkgreen', linewidth=2.5, label='Valor Total')
ax3.axhline(y=capital_inicial, color='red', linestyle='--', 
           linewidth=2, alpha=0.7, label=f'Capital Inicial: ${capital_inicial:,.0f}')

ax3.set_title('Evolución del Valor Total del Portafolio (180 días)', 
             fontsize=14, fontweight='bold')
ax3.set_xlabel('Fecha', fontsize=12)
ax3.set_ylabel('Valor del Portafolio (USD)', fontsize=12)
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:,.0f}'))

# Formato de fechas en eje x
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax3.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\n=== GRÁFICOS GENERADOS ===")
print("1. Evolución de Pesos por Activo")
print("2. Evolución de Pesos por Sector (stacked area)")
print("3. Evolución del Valor Total del Portafolio")

In [0]:
# ==============================================================================
# RESUMEN Y ESTADÍSTICAS FINALES
# ==============================================================================

import pandas as pd
import numpy as np

# Filtrar datos válidos (sin NaT)
df_valid = df_evolucion[df_evolucion.index.notna()].copy()

print("=== DATAFRAME df_evolucion - VISTA RESUMIDA ===")
print(f"\nColumnas principales del dataframe:")
print(f"  - Precios: {assets}")
print(f"  - Valores: {[f'{a}_Valor' for a in assets]}")
print(f"  - Pesos: {[f'{a}_Peso' for a in assets]}")
print(f"  - Sectores: {[f'Peso_Sector_{s}' for s in sectores_unicos]}")
print(f"  - Valor_Total_Portafolio")

print("\n=== MUESTRA DEL DATAFRAME (primeros 5 días) ===")
# Seleccionar columnas más importantes para mostrar
cols_muestra = ['AAPL.O_Peso', 'MSFT.O_Peso', 'AMZN.O_Peso', 
                'Peso_Sector_Information Technology', 
                'Peso_Sector_Consumer Discretionary',
                'Valor_Total_Portafolio']
display(df_valid[cols_muestra].head())

print("\n=== ESTADÍSTICAS DE DRIFT (Desviación de pesos iniciales) ===")
print("\nCuánto se han desviado los pesos de cada activo respecto a la asignación inicial:\n")

for i, activo in enumerate(assets):
    if w_optimo[i] > 0.001:  # Solo para activos con peso significativo
        peso_col = f'{activo}_Peso'
        drift = df_valid[peso_col] - w_optimo[i]
        
        print(f"{activo}:")
        print(f"  Peso Inicial: {w_optimo[i]:.4f} ({w_optimo[i]*100:.2f}%)")
        print(f"  Peso Final:   {df_valid[peso_col].iloc[-1]:.4f} ({df_valid[peso_col].iloc[-1]*100:.2f}%)")
        print(f"  Drift Máximo: {drift.max():.4f} ({drift.max()*100:.2f}%)")
        print(f"  Drift Mínimo: {drift.min():.4f} ({drift.min()*100:.2f}%)")
        print(f"  Drift Medio:  {drift.mean():.4f} ({drift.mean()*100:.2f}%)")
        print(f"  Volatilidad del Drift: {drift.std():.4f} ({drift.std()*100:.2f}%)\n")

print("\n=== ESTADÍSTICAS DE PESOS SECTORIALES ===")
for sector in sectores_unicos:
    peso_col = f'Peso_Sector_{sector}'
    print(f"\n{sector}:")
    print(f"  Peso Inicial: {df_valid[peso_col].iloc[0]:.4f} ({df_valid[peso_col].iloc[0]*100:.2f}%)")
    print(f"  Peso Final:   {df_valid[peso_col].iloc[-1]:.4f} ({df_valid[peso_col].iloc[-1]*100:.2f}%)")
    print(f"  Peso Máximo:  {df_valid[peso_col].max():.4f} ({df_valid[peso_col].max()*100:.2f}%)")
    print(f"  Peso Mínimo:  {df_valid[peso_col].min():.4f} ({df_valid[peso_col].min()*100:.2f}%)")
    print(f"  Promedio:     {df_valid[peso_col].mean():.4f} ({df_valid[peso_col].mean()*100:.2f}%)")

print("\n=== MÉTRICAS DE RENDIMIENTO DEL PORTAFOLIO ===")
valor_inicial = df_valid['Valor_Total_Portafolio'].iloc[0]
valor_final = df_valid['Valor_Total_Portafolio'].iloc[-1]
retorno_total = (valor_final / valor_inicial - 1)
retornos_diarios = df_valid['Valor_Total_Portafolio'].pct_change().dropna()

print(f"\nValor Inicial:        ${valor_inicial:,.2f}")
print(f"Valor Final:          ${valor_final:,.2f}")
print(f"Retorno Total:        {retorno_total*100:.2f}%")
print(f"Retorno Anualizado:   {(((1 + retorno_total)**(252/len(df_valid))) - 1)*100:.2f}%")
print(f"Volatilidad Diaria:   {retornos_diarios.std()*100:.2f}%")
print(f"Volatilidad Anualizada: {retornos_diarios.std()*np.sqrt(252)*100:.2f}%")
print(f"Sharpe Ratio (rf=4%): {((((1 + retorno_total)**(252/len(df_valid))) - 1) - 0.04) / (retornos_diarios.std()*np.sqrt(252)):.2f}")
print(f"\nMáximo Drawdown:     {((df_valid['Valor_Total_Portafolio'] / df_valid['Valor_Total_Portafolio'].cummax()) - 1).min()*100:.2f}%")

print("\n" + "="*80)
print("DATAFRAME 'df_evolucion' DISPONIBLE PARA ANÁLISIS ADICIONALES")
print("="*80)